In [ ]:
import os
import shutil
import marimo as mo

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, SequentialLR, LinearLR

from tqdm import tqdm

from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

from monai.transforms import (
    Compose,
    RandFlipd,
    RandRotate90d,
    RandAffined,
    RandGaussianNoised,
    RandAdjustContrastd,
    RandGaussianSmoothd,
)
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR
from monai.networks.utils import one_hot
from monai.inferers import sliding_window_inference

import wandb

In [ ]:
NUM_LABELS = 4
PATCH_SIZE = (96, 96, 96)
LR = 1e-4
BATCH_SIZE = 16
EPOCHS = 50

DATASET_REPO = "AG2307/pengwin-2026-anatomy-segmentation-pelvic"
HF_REPO = "AG2307/pengwin-2026"

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
LOCAL_CHECKPOINT = f"{CHECKPOINT_DIR}/best_model_pelvic.pth"

In [ ]:
class PelvicDataset(Dataset):
    def __init__(self, hf_dataset, transforms=None):
        self.dataset = hf_dataset
        self.transforms = transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        if self.transforms:
            sample = self.transforms(sample)
        return sample


def build_transforms():
    return Compose(
        [
            RandAffined(
                keys=["image", "label"],
                prob=0.3,
                rotate_range=(0.15, 0.15, 0.15),
                scale_range=(0.1, 0.1, 0.1),
                translate_range=(10, 10, 5),
                mode=("bilinear", "nearest"),
                padding_mode="border",
            ),
            RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.1),
            RandAdjustContrastd(keys=["image"], prob=0.3, gamma=(0.7, 1.3)),
            RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5, 1.0)),
        ]
    )


def build_dataloaders():
    data = load_dataset(DATASET_REPO)
    data = data.remove_columns(
        [
            "patient_id",
            "original_spacing",
            "original_shape",
            "original_affine",
        ]
    )
    data = data.with_format("torch")

    train_dataset = PelvicDataset(data["train"], transforms=build_transforms())
    val_dataset = PelvicDataset(data["validation"], transforms=None)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    return train_loader, val_loader

In [ ]:
def maybe_download_checkpoint(api, hf_token):
    # remove any stale local checkpoint from a previous session
    if os.path.exists(LOCAL_CHECKPOINT):
        os.remove(LOCAL_CHECKPOINT)
        print("Removed stale local checkpoint.")

    try:
        downloaded = hf_hub_download(
            repo_id=HF_REPO,
            filename="best_model_pelvic.pth",
            repo_type="model",
            token=hf_token,
        )
        shutil.copy(downloaded, LOCAL_CHECKPOINT)
        print("Downloaded checkpoint from HuggingFace.")
    except Exception as e:
        print(f"No remote checkpoint found: {e}")


def load_checkpoint(model, optimizer, scheduler):
    """Restore state from a local checkpoint if present. Returns (start_epoch, best_dice)."""
    if not os.path.exists(LOCAL_CHECKPOINT):
        print("No checkpoint found, starting fresh...")
        return 1, 0.0, None

    print("Found existing checkpoint, loading...")
    ckpt = torch.load(LOCAL_CHECKPOINT, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_dice = ckpt["best_dice"]
    wandb_run_id = ckpt.get("wandb_run_id", None)
    print(f"  Resuming from epoch {start_epoch}, best Dice: {best_dice:.4f}")
    return start_epoch, best_dice, wandb_run_id


def save_checkpoint(
    model, optimizer, scheduler, epoch, best_dice, api, hf_token
):
    """Save locally and push to HF (main process only — caller must guard)."""
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_dice": best_dice,
            "wandb_run_id": wandb.run.id,
        },
        LOCAL_CHECKPOINT,
    )

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, scheduler, device):
    model.train()
    total_loss = 0.0
    num_batches = len(loader)

    pbar = tqdm(loader, desc="Training")
    for i, batch in enumerate(pbar):
        image = batch["image"].to(device)
        label = batch["label"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred = model(image)
            loss = loss_fn(pred, label)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()

        total_loss += loss.item()

        if i % 5 == 0:
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                avg_loss=f"{total_loss / (i + 1):.4f}",
            )

    return total_loss / num_batches


def evaluate(model, loader, loss_fn, device):
    """Runs on the main process only (caller guards). Returns (val_loss, val_dice)."""
    dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
    model.eval()

    total_loss = 0.0
    num_batches = len(loader)

    pbar = tqdm(loader, desc="Validation")
    for i, batch in enumerate(pbar):
        image = batch["image"].to(device)
        label = batch["label"].to(device)

        with (
            torch.no_grad(),
            torch.autocast(device_type="cuda", dtype=torch.bfloat16),
        ):
            pred = sliding_window_inference(
                inputs=image,
                roi_size=PATCH_SIZE,
                sw_batch_size=4,
                predictor=model,
                overlap=0.5,
                mode="gaussian",
            )

            total_loss += loss_fn(pred, label).item()

            pred_hard = torch.argmax(pred, dim=1, keepdim=True)
            dice_metric(
                one_hot(pred_hard, num_classes=NUM_LABELS),
                one_hot(label, num_classes=NUM_LABELS),
            )

        if i % 5 == 0:
            pbar.set_postfix(loss=f"{total_loss / (i + 1):.4f}")

    val_loss = total_loss / num_batches
    per_class = dice_metric.aggregate()
    val_dice = per_class.mean().item()
    dice_metric.reset()
    return val_loss, val_dice, per_class

In [ ]:
def build_model_and_optim(num_training_steps):
    model = SwinUNETR(
        in_channels=1,
        out_channels=NUM_LABELS,
        feature_size=48,
        use_v2=True,
        use_checkpoint=True,
    )
    loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)

    warmup_steps = num_training_steps // 10
    warmup = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = CosineAnnealingLR(
        optimizer, T_max=num_training_steps - warmup_steps
    )
    scheduler = SequentialLR(
        optimizer, [warmup, cosine], milestones=[warmup_steps]
    )

    return model, loss_fn, optimizer, scheduler

In [ ]:
def main():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    hf_token = os.environ.get("HF_TOKEN")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    api = HfApi()
    api.create_repo(
        repo_id=HF_REPO,
        repo_type="model",
        private=True,
        token=hf_token,
        exist_ok=True,
    )

    train_loader, val_loader = build_dataloaders()
    num_training_steps = len(train_loader) * EPOCHS

    model, loss_fn, optimizer, scheduler = build_model_and_optim(
        num_training_steps
    )
    model = model.to(device)

    maybe_download_checkpoint(api, hf_token)
    start_epoch, best_dice, wandb_run_id = load_checkpoint(
        model, optimizer, scheduler
    )

    # model = torch.compile(model)

    wandb.init(
        project="pengwin-2026-pelvic",
        id=wandb_run_id,
        resume="allow" if wandb_run_id else None,
        config={
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "patch_size": PATCH_SIZE[0],
            "num_classes": NUM_LABELS,
        },
    )

    CLASS_NAMES = ["sacrum", "left_hipbone", "right_hipbone"]

    for epoch in range(start_epoch, EPOCHS + 1):
        print(f"Epoch {epoch}")
        train_loss = train_one_epoch(
            model, train_loader, optimizer, loss_fn, scheduler, device
        )
        val_loss, val_dice, per_class = evaluate(
            model, val_loader, loss_fn, device
        )

        class_str = " | ".join(
            f"{name}: {per_class[i]:.4f}" for i, name in enumerate(CLASS_NAMES)
        )
        print(
            f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}"
        )
        print(f"  Per-class: {class_str}")

        log_dict = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_dice": val_dice,
            "lr": optimizer.param_groups[0]["lr"],
        }
        for i, name in enumerate(CLASS_NAMES):
            log_dict[f"dice_{name}"] = per_class[i].item()

        wandb.log(log_dict)

        save_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            best_dice,
            api,
            hf_token,
        )

        api.upload_file(
            path_or_fileobj=LOCAL_CHECKPOINT,
            path_in_repo="latest_model_pelvic.pth",
            repo_id=HF_REPO,
            repo_type="model",
            token=hf_token,
            commit_message=f"Epoch {epoch} | Dice: {val_dice:.4f}",
        )
        print("  Pushed current model to HuggingFace.")

        if val_dice > best_dice:
            best_dice = val_dice
            print(f"  New best! Dice: {best_dice:.4f}")

            api.upload_file(
                path_or_fileobj=LOCAL_CHECKPOINT,
                path_in_repo="best_model_pelvic.pth",
                repo_id=HF_REPO,
                repo_type="model",
                token=hf_token,
                commit_message=f"Epoch {epoch} | Dice: {best_dice:.4f}",
            )
            print("  Pushed to HuggingFace.")

    wandb.finish()


if __name__ == "__main__":
    main()


Generating validation split: 100%|██████████| 34/34 [00:28<00:00,  1.21 examples/s]


No remote checkpoint found: 404 Client Error. (Request ID: Root=1-6a3e5a28-3aaefe4525006f7553baf6a2;f4c2fd47-ac82-4f0f-9291-33e5ffb1522d)

Entry Not Found for url: https://huggingface.co/AG2307/pengwin-2026/resolve/main/best_model_pelvic.pth.
No checkpoint found, starting fresh...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ag2307 (ag2307-university-college-dublin) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.27.1
wandb: Run data is saved locally in /marimo/wandb/run-20260626_105328-8jjdb7fs
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run upbeat-mountain-2
wandb: ⭐️ View project at https://wandb.ai/ag2307-university-college-dublin/pengwin-2026-pelvic
wandb: 🚀 View run at https://wandb.ai/ag2307-university-college-dublin/pengwin-2026-pelvic/runs/8jjdb7fs


Epoch 1


Validation:   0%|          | 0/34 [00:00<?, ?it/s]/tmp/uv-venv/lib/python3.13/site-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/tmp/uv-venv/lib/python3.13/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/

...

  Train Loss: 0.5152 | Val Loss: 0.3208 | Val Dice: 0.9455
  Per-class: sacrum: 0.9288 | left_hipbone: 0.9548 | right_hipbone: 0.9530


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...nts/best_model_pelvic.pth:   0%|          | 26.9kB /  881MB            

Processing Files (0 / 1)      :   0%|          | 26.9kB /  881MB,   ???B/s  

Processing Files (0 / 1)      :   7%|▋         | 62.5MB /  881MB,   ???B/s  
New Data Upload               :  47%|████▋     | 62.5MB /  134MB,   ???B/s  

Processing Files (0 / 1)      :  11%|█         | 97.0MB /  881MB, 9.37MB/s  
New Data Upload               :  48%|████▊     | 96.9MB /  201MB, 9.37MB/s  

Processing Files (0 / 1)      :  15%|█▌        |  134MB /  881MB, 12.7MB/s  
New Data Upload               :  66%|██████▋   |  134MB /  201MB, 12.7MB/s  

Processing Files (0 / 1)      :  15%|█▌        |  134MB /  881MB, 12.4MB/s  
New Data Upload               :  50%|████▉     |  134MB /  268MB, 12.4MB/s  

Processing Files (0 / 1)      :  21%|██▏       |  188MB /  881MB, 17.3MB/s  


  Pushed current model to HuggingFace.
Epoch 39


Validation:  29%|██▉       | 10/34 [02:43<07:04, 17.69s/it, loss=0.2930]

exception: This cell was interrupted and needs to be re-run